# Previsione Consegne Materie Prime - Short Notebook 1

**Nomi:** [Inserire nomi]

**ID Studenti:** [Inserire ID]

**Team Name Kaggle:** [Inserire team name]

---

Questo notebook implementa un modello di previsione per le consegne cumulative di materie prime utilizzando **solo il dataset receivals.csv**.

**Obiettivo**: Prevedere il peso cumulativo di ogni `rm_id` dal 1 Gennaio al 31 Maggio 2025.

**Metrica**: Quantile Loss 0.2 (penalizza maggiormente le sovrastime).

**Approccio**: Time series forecasting con LightGBM usando regressione quantile.

## 1. Import Librerie e Caricamento Dati

In [14]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Caricamento dei dati
print("Caricamento dei dati...")
receivals_df = pd.read_csv('data/kernel/receivals.csv')
prediction_mapping_df = pd.read_csv('data/prediction_mapping.csv')

# Conversione delle date
receivals_df['date_arrival'] = pd.to_datetime(receivals_df['date_arrival'], errors='coerce', utc=True)
receivals_df = receivals_df.dropna(subset=['date_arrival'])
# Rimuovi il fuso orario e converti in data locale
receivals_df['date'] = receivals_df['date_arrival'].dt.tz_localize(None)

print(f"Receivals shape: {receivals_df.shape}")
print(f"Date range: {receivals_df['date'].min()} to {receivals_df['date'].max()}")
print(f"Unique rm_ids: {receivals_df['rm_id'].nunique()}")

Caricamento dei dati...
Receivals shape: (122590, 11)
Date range: 2004-06-15 11:34:00 to 2024-12-19 13:36:00
Unique rm_ids: 203
Receivals shape: (122590, 11)
Date range: 2004-06-15 11:34:00 to 2024-12-19 13:36:00
Unique rm_ids: 203


## 2. Aggregazione Giornaliera delle Consegne

Aggreghiamo i pesi netti giornalieri per ogni `rm_id`.

In [15]:
# Aggregazione giornaliera
daily_receivals = receivals_df.groupby(['rm_id', receivals_df['date'].dt.date])['net_weight'].sum().reset_index()
daily_receivals.columns = ['rm_id', 'date', 'net_weight']
daily_receivals['date'] = pd.to_datetime(daily_receivals['date'])

print(f"Daily receivals shape: {daily_receivals.shape}")
print(f"\nPrime 5 righe:")
print(daily_receivals.head())

Daily receivals shape: (41933, 3)

Prime 5 righe:
   rm_id       date  net_weight
0  342.0 2004-06-23     24940.0
1  343.0 2005-03-29     21760.0
2  345.0 2004-09-01     22780.0
3  346.0 2004-06-24       820.0
4  346.0 2004-06-30     21260.0


## 3. Creazione del Master Table

Creiamo una griglia completa di tutte le combinazioni `(rm_id, data)` dal minimo storico fino al 31 Dicembre 2024.

In [16]:
# Estrazione rm_ids unici e date range
unique_rm_ids = receivals_df['rm_id'].unique()
start_date = daily_receivals['date'].min()
end_date = pd.Timestamp('2024-12-31')
date_range = pd.date_range(start=start_date, end=end_date, freq='D')

print(f"Creazione master table per {len(unique_rm_ids)} rm_ids e {len(date_range)} giorni...")

# Creazione master table usando MultiIndex per efficienza
multi_index = pd.MultiIndex.from_product([unique_rm_ids, date_range], names=['rm_id', 'date'])
master_df = pd.DataFrame(index=multi_index).reset_index()

# Merge con le consegne effettive
master_df = pd.merge(master_df, daily_receivals, on=['rm_id', 'date'], how='left')
master_df['net_weight'] = master_df['net_weight'].fillna(0)

print(f"Master table shape: {master_df.shape}")

Creazione master table per 204 rm_ids e 7505 giorni...
Master table shape: (1531020, 3)


## 4. Feature Engineering

Creiamo feature temporali e statistiche basate sullo storico delle consegne.

In [17]:
print("Creazione feature temporali...")

# Feature temporali
master_df['year'] = master_df['date'].dt.year
master_df['month'] = master_df['date'].dt.month
master_df['day'] = master_df['date'].dt.day
master_df['dayofweek'] = master_df['date'].dt.dayofweek
master_df['dayofyear'] = master_df['date'].dt.dayofyear
master_df['weekofyear'] = master_df['date'].dt.isocalendar().week.astype(int)
master_df['quarter'] = master_df['date'].dt.quarter

# Ordinamento per calcolo delle feature lag/rolling
master_df = master_df.sort_values(['rm_id', 'date'])

print("Creazione feature lag e rolling in parallelo...")
from joblib import Parallel, delayed

def compute_features_for_rm(rm_data):
    """Calcola feature lag e rolling per un singolo rm_id"""
    rm_data = rm_data.sort_values('date').copy()
    
    # Feature Lag (ridotte a 3)
    for lag in [7, 28, 56]:
        rm_data[f'lag_{lag}d'] = rm_data['net_weight'].shift(lag)
    
    # Feature Rolling (solo mean e sum per velocità, ridotte a 2 finestre)
    for window in [7, 28]:
        rolling_obj = rm_data['net_weight'].shift(1).rolling(window, min_periods=1)
        rm_data[f'rolling_mean_{window}d'] = rolling_obj.mean()
        rm_data[f'rolling_sum_{window}d'] = rolling_obj.sum()
    
    return rm_data

# Parallelizzazione per rm_id (usa tutti i core disponibili)
n_jobs = -1
results = Parallel(n_jobs=n_jobs, verbose=1)(
    delayed(compute_features_for_rm)(group) 
    for _, group in master_df.groupby('rm_id')
)

# Ricombina i risultati
master_df = pd.concat(results, ignore_index=True).sort_values(['rm_id', 'date'])

# Statistiche complessive per rm_id
print("Creazione statistiche aggregate per rm_id...")
rm_stats = master_df.groupby('rm_id')['net_weight'].agg([
    ('rm_mean', 'mean'),
    ('rm_std', 'std'),
    ('rm_median', 'median')
]).reset_index()
master_df = pd.merge(master_df, rm_stats, on='rm_id', how='left')

print(f"Master table con feature: {master_df.shape}")
print(f"Feature create: {master_df.columns.tolist()}")

Creazione feature temporali...
Creazione feature lag e rolling in parallelo...
Creazione feature lag e rolling in parallelo...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 11 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    0.1s


Creazione statistiche aggregate per rm_id...
Master table con feature: (1523515, 20)
Feature create: ['rm_id', 'date', 'net_weight', 'year', 'month', 'day', 'dayofweek', 'dayofyear', 'weekofyear', 'quarter', 'lag_7d', 'lag_28d', 'lag_56d', 'rolling_mean_7d', 'rolling_sum_7d', 'rolling_mean_28d', 'rolling_sum_28d', 'rm_mean', 'rm_std', 'rm_median']


[Parallel(n_jobs=-1)]: Done 203 out of 203 | elapsed:    0.3s finished


## 5. Preparazione Dataset per Training

Dividiamo i dati in training e validazione basandoci sulla timeline.

In [18]:
# Definizione feature e target (SENZA rm_id come feature categoriale)
# Le statistiche rm_mean, rm_std, rm_median già catturano le info specifiche per rm_id
feature_cols = [
    'year', 'month', 'day', 'dayofweek', 'dayofyear', 'weekofyear', 'quarter',
    'lag_7d', 'lag_28d', 'lag_56d',
    'rolling_mean_7d', 'rolling_sum_7d',
    'rolling_mean_28d', 'rolling_sum_28d',
    'rm_mean', 'rm_std', 'rm_median'
]
target_col = 'net_weight'

# Split temporale: train fino a fine 2023, validation su 2024
train_df = master_df[master_df['date'] < '2024-01-01'].copy()
val_df = master_df[master_df['date'] >= '2024-01-01'].copy()

# Rimozione righe con NaN nelle feature (dai primi giorni senza lag/rolling)
train_df = train_df.dropna(subset=feature_cols)
val_df = val_df.dropna(subset=feature_cols)

# Separazione feature e target (SENZA rm_id)
X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_val = val_df[feature_cols]
y_val = val_df[target_col]

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Feature utilizzate: {feature_cols}")

Training set: (1437849, 17)
Validation set: (74298, 17)
Feature utilizzate: ['year', 'month', 'day', 'dayofweek', 'dayofyear', 'weekofyear', 'quarter', 'lag_7d', 'lag_28d', 'lag_56d', 'rolling_mean_7d', 'rolling_sum_7d', 'rolling_mean_28d', 'rolling_sum_28d', 'rm_mean', 'rm_std', 'rm_median']


## 6. Training del Modello LightGBM

Addestriamo un modello LightGBM con regressione quantile (alpha=0.2) per ottenere previsioni conservative.

In [19]:
print("Training del modello LightGBM...")

# Parametri del modello (MODIFICATO: alpha=0.5 per previsioni mediane invece di troppo conservative)
params = {
    'objective': 'quantile',
    'alpha': 0.5,  # Quantile 0.5 (mediana) per bilanciare sovra/sottostime
    'metric': 'quantile',
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

model = lgb.LGBMRegressor(**params)

# Training con early stopping (SENZA categorical_feature perché abbiamo rimosso rm_id)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='quantile',
    callbacks=[lgb.early_stopping(100, verbose=False)]
)

print(f"Training completato. Best iteration: {model.best_iteration_}")

# Valutazione sul validation set
val_pred = model.predict(X_val)
val_pred = np.clip(val_pred, 0, None)  # Non ammettere previsioni negative

# Calcolo quantile loss custom
errors = y_val - val_pred
quantile_loss = np.where(errors >= 0, 0.2 * errors, -0.8 * errors)
mean_quantile_loss = quantile_loss.mean()

print(f"\nValidation Quantile Loss (0.2): {mean_quantile_loss:.4f}")

Training del modello LightGBM...
Training completato. Best iteration: 49

Validation Quantile Loss (0.2): 288.1272
Training completato. Best iteration: 49

Validation Quantile Loss (0.2): 288.1272


## 7. Predizione per il 2025

Creiamo le previsioni per il periodo Gennaio-Maggio 2025.

In [20]:
print("Creazione set di test per il 2025...")

# Date per il 2025
test_date_range = pd.date_range(start='2025-01-01', end='2025-05-31', freq='D')
test_multi_index = pd.MultiIndex.from_product([unique_rm_ids, test_date_range], names=['rm_id', 'date'])
test_df = pd.DataFrame(index=test_multi_index).reset_index()

# IMPORTANTE: Invece di usare 0 come placeholder, usiamo NaN per distinguere valori mancanti da zero reale
# Questo evita di propagare zeri falsi nelle feature lag/rolling
test_df['net_weight'] = np.nan  
combined_df = pd.concat([master_df, test_df], ignore_index=True)
combined_df = combined_df.sort_values(['rm_id', 'date'])

# Riempi i NaN del test set con 0 SOLO DOPO aver calcolato master_df (che ha dati reali)
# In questo modo i lag prenderanno i valori storici corretti da master_df
combined_df['net_weight'] = combined_df['net_weight'].fillna(0)

print("Creazione feature per il test set...")

# NON ricalcolare le feature temporali - sono già presenti nel test_df
# Creiamo solo le feature temporali per il test set 2025
test_df['year'] = test_df['date'].dt.year
test_df['month'] = test_df['date'].dt.month
test_df['day'] = test_df['date'].dt.day
test_df['dayofweek'] = test_df['date'].dt.dayofweek
test_df['dayofyear'] = test_df['date'].dt.dayofyear
test_df['weekofyear'] = test_df['date'].dt.isocalendar().week.astype(int)
test_df['quarter'] = test_df['date'].dt.quarter

print("Creazione feature lag e rolling per il test set (usando dati storici reali)...")

def compute_test_features_for_rm(rm_data):
    """Calcola feature lag e rolling per un singolo rm_id (include dati storici + 2025)"""
    rm_data = rm_data.sort_values('date').copy()
    
    # Feature Lag - IMPORTANTE: usa shift su net_weight che contiene dati storici reali
    for lag in [7, 28, 56]:
        rm_data[f'lag_{lag}d'] = rm_data['net_weight'].shift(lag)
    
    # Feature Rolling - IMPORTANTE: usa rolling su net_weight che contiene dati storici reali  
    for window in [7, 28]:
        rolling_obj = rm_data['net_weight'].shift(1).rolling(window, min_periods=1)
        rm_data[f'rolling_mean_{window}d'] = rolling_obj.mean()
        rm_data[f'rolling_sum_{window}d'] = rolling_obj.sum()
    
    return rm_data

# Parallelizzazione per rm_id (usa tutti i core disponibili)
# IMPORTANTE: Passiamo combined_df completo (con dati storici reali fino a 2024-12-31)
results = Parallel(n_jobs=-1, verbose=1)(
    delayed(compute_test_features_for_rm)(group[['rm_id', 'date', 'net_weight']]) 
    for _, group in combined_df.groupby('rm_id')
)

# Ricombina i risultati
combined_df = pd.concat(results, ignore_index=True).sort_values(['rm_id', 'date'])

# Aggiungi le feature temporali al combined_df completo
combined_df = combined_df.merge(
    pd.concat([
        master_df[['rm_id', 'date', 'year', 'month', 'day', 'dayofweek', 'dayofyear', 'weekofyear', 'quarter']],
        test_df[['rm_id', 'date', 'year', 'month', 'day', 'dayofweek', 'dayofyear', 'weekofyear', 'quarter']]
    ]),
    on=['rm_id', 'date'],
    how='left'
)

# Aggiungi le statistiche rm_id (dalle stesse già calcolate nel master_df)
combined_df = pd.merge(combined_df, rm_stats, on='rm_id', how='left')

# Estrazione solo del test set 2025
final_test_df = combined_df[combined_df['date'].dt.year == 2025].copy()

# Verifica che le colonne statistiche siano presenti
print(f"Colonne disponibili: {final_test_df.columns.tolist()}")

# Verifica NaN nelle feature (dovrebbero essere minimi ora che usiamo dati storici reali)
print(f"\nVerifica NaN per feature:")
for col in feature_cols:
    nan_count = final_test_df[col].isna().sum()
    if nan_count > 0:
        print(f"  {col}: {nan_count} NaN ({100*nan_count/len(final_test_df):.2f}%)")

# Riempi eventuali NaN SOLO se necessario (es. primi giorni gennaio senza 56 giorni di storia)
lag_rolling_cols = [c for c in feature_cols if 'lag_' in c or 'rolling_' in c]
for col in lag_rolling_cols:
    if final_test_df[col].isna().any():
        print(f"  Riempiendo NaN in {col} con 0...")
        final_test_df[col] = final_test_df[col].fillna(0)

# Verifica statistiche delle feature lag nel test set (dovrebbero essere > 0 ora)
print(f"\nStatistiche feature lag nel test set (dopo correzione):")
for col in ['lag_7d', 'lag_28d', 'lag_56d']:
    print(f"  {col}: mean={final_test_df[col].mean():.2f}, std={final_test_df[col].std():.2f}, max={final_test_df[col].max():.2f}")

# Rimuovi righe con NaN nelle statistiche rm_id o altre feature critiche
final_test_df = final_test_df.dropna(subset=['rm_mean', 'rm_std', 'rm_median'])

print(f"Test set 2025 shape: {final_test_df.shape}")

Creazione set di test per il 2025...
Creazione feature per il test set...
Creazione feature lag e rolling per il test set (usando dati storici reali)...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 11 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 203 out of 203 | elapsed:    0.3s finished
[Parallel(n_jobs=-1)]: Done 203 out of 203 | elapsed:    0.3s finished


Colonne disponibili: ['rm_id', 'date', 'net_weight', 'lag_7d', 'lag_28d', 'lag_56d', 'rolling_mean_7d', 'rolling_sum_7d', 'rolling_mean_28d', 'rolling_sum_28d', 'year', 'month', 'day', 'dayofweek', 'dayofyear', 'weekofyear', 'quarter', 'rm_mean', 'rm_std', 'rm_median']

Verifica NaN per feature:

Statistiche feature lag nel test set (dopo correzione):
  lag_7d: mean=0.00, std=0.00, max=0.00
  lag_28d: mean=116.63, std=2847.97, max=183690.00
  lag_56d: mean=348.41, std=4498.10, max=183690.00
Test set 2025 shape: (30653, 20)


## 8. Generazione delle Predizioni Giornaliere

In [21]:
print("Generazione predizioni giornaliere per il 2025...")

# Preparazione feature per la predizione (SENZA rm_id)
X_test = final_test_df[feature_cols].copy()

# Predizione
daily_predictions = model.predict(X_test)
daily_predictions = np.clip(daily_predictions, 0, None)  # No valori negativi

# Assegna predizioni giornaliere
final_test_df['predicted_daily_weight'] = daily_predictions

# Calcolo pesi cumulativi immediatamente
final_test_df = final_test_df.sort_values(['rm_id', 'date'])
final_test_df['predicted_cumulative_weight'] = final_test_df.groupby('rm_id')['predicted_daily_weight'].cumsum()

# Converti rm_id in int per il merge con prediction_mapping
final_test_df['rm_id'] = final_test_df['rm_id'].astype(int)

print(f"Predizioni generate: {len(daily_predictions)}")
print(f"Range predizioni giornaliere: [{daily_predictions.min():.2f}, {daily_predictions.max():.2f}]")
print(f"Range predizioni cumulative: [{final_test_df['predicted_cumulative_weight'].min():.2f}, {final_test_df['predicted_cumulative_weight'].max():.2f}]")
print(f"\nPrime 20 predizioni:")
print(final_test_df[['rm_id', 'date', 'predicted_daily_weight', 'predicted_cumulative_weight']].head(20))

Generazione predizioni giornaliere per il 2025...
Predizioni generate: 30653
Range predizioni giornaliere: [0.00, 7946.62]
Range predizioni cumulative: [0.00, 48532.88]

Prime 20 predizioni:
      rm_id       date  predicted_daily_weight  predicted_cumulative_weight
7505    342 2025-01-01                     0.0                          0.0
7506    342 2025-01-02                     0.0                          0.0
7507    342 2025-01-03                     0.0                          0.0
7508    342 2025-01-04                     0.0                          0.0
7509    342 2025-01-05                     0.0                          0.0
7510    342 2025-01-06                     0.0                          0.0
7511    342 2025-01-07                     0.0                          0.0
7512    342 2025-01-08                     0.0                          0.0
7513    342 2025-01-09                     0.0                          0.0
7514    342 2025-01-10                     0.0   

## 9. Calcolo delle Predizioni Cumulative

Calcoliamo il peso cumulativo per ogni `rm_id` dal 1 Gennaio in poi.

## 10. Creazione File di Submission

Creiamo il file `submission.csv` nel formato richiesto.

## 10.1. Debug: Verifica Dati Prima della Submission

In [22]:
print("Debug: Verifica dati prima del merge...")

# Verifica prediction_mapping
print(f"\nPrediction mapping shape: {prediction_mapping_df.shape}")
print(f"Prime righe prediction_mapping:")
print(prediction_mapping_df.head(10))

# Verifica final_test_df
print(f"\nFinal test df shape: {final_test_df.shape}")
print(f"Prime righe final_test_df:")
print(final_test_df[['rm_id', 'date', 'predicted_daily_weight', 'predicted_cumulative_weight']].head(20))

# Verifica rm_id specifico
test_rm = 365
print(f"\nDati per rm_id={test_rm}:")
test_data = final_test_df[final_test_df['rm_id'] == test_rm].head(10)
print(test_data[['rm_id', 'date', 'predicted_cumulative_weight']])

# Verifica mapping per stesso rm_id
print(f"\nMapping per rm_id={test_rm}:")
mapping_data = prediction_mapping_df[prediction_mapping_df['rm_id'] == test_rm].head(10)
print(mapping_data)

Debug: Verifica dati prima del merge...

Prediction mapping shape: (30450, 4)
Prime righe prediction_mapping:
   ID  rm_id forecast_start_date forecast_end_date
0   1    365          2025-01-01        2025-01-02
1   2    365          2025-01-01        2025-01-03
2   3    365          2025-01-01        2025-01-04
3   4    365          2025-01-01        2025-01-05
4   5    365          2025-01-01        2025-01-06
5   6    365          2025-01-01        2025-01-07
6   7    365          2025-01-01        2025-01-08
7   8    365          2025-01-01        2025-01-09
8   9    365          2025-01-01        2025-01-10
9  10    365          2025-01-01        2025-01-11

Final test df shape: (30653, 22)
Prime righe final_test_df:
      rm_id       date  predicted_daily_weight  predicted_cumulative_weight
7505    342 2025-01-01                     0.0                          0.0
7506    342 2025-01-02                     0.0                          0.0
7507    342 2025-01-03                  

In [23]:
print("Analisi dettagliata delle predizioni...")

# Quali rm_id hanno predizioni non-zero?
nonzero_rm_ids = final_test_df[final_test_df['predicted_daily_weight'] > 0]['rm_id'].unique()
print(f"\nNumero di rm_id con predizioni non-zero: {len(nonzero_rm_ids)}")
print(f"Primi 10 rm_id con predizioni: {sorted(nonzero_rm_ids)[:10]}")

# Verifica se rm_id=365 è nel final_test_df
print(f"\nrm_id=365 presente in final_test_df: {365 in final_test_df['rm_id'].values}")
print(f"Tipo di rm_id in final_test_df: {final_test_df['rm_id'].dtype}")
print(f"Tipo di rm_id in prediction_mapping: {prediction_mapping_df['rm_id'].dtype}")

# Vediamo un rm_id con predizioni non-zero
if len(nonzero_rm_ids) > 0:
    test_rm_nonzero = nonzero_rm_ids[0]
    print(f"\nEsempio rm_id={test_rm_nonzero} con predizioni non-zero:")
    sample = final_test_df[final_test_df['rm_id'] == test_rm_nonzero].head(10)
    print(sample[['rm_id', 'date', 'predicted_daily_weight', 'predicted_cumulative_weight']])
    
# Quante righe matchano tra final_test_df e prediction_mapping?
# Prima converti forecast_end_date in datetime
test_mapping = prediction_mapping_df[['ID', 'rm_id', 'forecast_end_date']].head(100).copy()
test_mapping['forecast_end_date'] = pd.to_datetime(test_mapping['forecast_end_date'])

test_merge = pd.merge(
    test_mapping,
    final_test_df[['rm_id', 'date', 'predicted_cumulative_weight']],
    left_on=['rm_id', 'forecast_end_date'],
    right_on=['rm_id', 'date'],
    how='left'
)
print(f"\nTest merge (prime 100 righe mapping):")
print(f"Righe con match (non-NaN): {test_merge['predicted_cumulative_weight'].notna().sum()}")
print(f"Righe senza match (NaN): {test_merge['predicted_cumulative_weight'].isna().sum()}")

# Mostra qualche esempio del merge
print(f"\nPrime 10 righe del test merge:")
print(test_merge[['ID', 'rm_id', 'forecast_end_date', 'predicted_cumulative_weight']].head(10))

Analisi dettagliata delle predizioni...

Numero di rm_id con predizioni non-zero: 13
Primi 10 rm_id con predizioni: [2130, 2131, 2132, 2134, 2135, 2142, 2144, 3122, 3123, 3124]

rm_id=365 presente in final_test_df: True
Tipo di rm_id in final_test_df: int64
Tipo di rm_id in prediction_mapping: int64

Esempio rm_id=2130 con predizioni non-zero:
        rm_id       date  predicted_daily_weight  predicted_cumulative_weight
581705   2130 2025-01-01                0.000000                     0.000000
581706   2130 2025-01-02                0.000000                     0.000000
581707   2130 2025-01-03                0.000000                     0.000000
581708   2130 2025-01-04                0.000000                     0.000000
581709   2130 2025-01-05                0.000000                     0.000000
581710   2130 2025-01-06                0.000000                     0.000000
581711   2130 2025-01-07             7946.622926                  7946.622926
581712   2130 2025-01-08      

In [24]:
print("Analisi rm_id nel training vs test...")

# rm_id nel training (prendi da train_df, non da X_train che non ha più rm_id)
train_rm_ids = set(train_df['rm_id'].unique())
print(f"\nrm_id nel training: {len(train_rm_ids)}")

# rm_id nel test
test_rm_ids = set(final_test_df['rm_id'].unique())
print(f"rm_id nel test: {len(test_rm_ids)}")

# rm_id nel test che NON sono nel training
unseen_rm_ids = test_rm_ids - train_rm_ids
print(f"rm_id nel test ma NON nel training: {len(unseen_rm_ids)}")
if len(unseen_rm_ids) > 0:
    print(f"  Esempi: {sorted(list(unseen_rm_ids))[:10]}")

# rm_id in comune
common_rm_ids = test_rm_ids & train_rm_ids
print(f"rm_id in comune training/test: {len(common_rm_ids)}")

# Verifica predizioni per rm_id comuni
if len(common_rm_ids) > 0:
    common_sample = list(common_rm_ids)[:5]
    print(f"\nVerifica predizioni per 5 rm_id comuni ({common_sample}):")
    for rm in common_sample:
        pred_sum = final_test_df[final_test_df['rm_id'] == rm]['predicted_daily_weight'].sum()
        print(f"  rm_id={rm}: somma predizioni = {pred_sum:.2f}")

Analisi rm_id nel training vs test...

rm_id nel training: 203
rm_id nel test: 203
rm_id nel test ma NON nel training: 0
rm_id in comune training/test: 203

Verifica predizioni per 5 rm_id comuni ([2561.0, 4101.0, 2061.0, 3601.0, 3101.0]):
  rm_id=2561.0: somma predizioni = 0.00
  rm_id=4101.0: somma predizioni = 0.00
  rm_id=2061.0: somma predizioni = 0.00
  rm_id=3601.0: somma predizioni = 0.00
  rm_id=3101.0: somma predizioni = 0.00


In [25]:
print("Debug approfondito: analisi feature del test set...")

# Verifica feature di un rm_id comune
sample_rm = 2561.0
print(f"\n=== Analisi rm_id={sample_rm} ===")

# Feature del test set per questo rm_id
test_sample = final_test_df[final_test_df['rm_id'] == sample_rm].head(10)
print(f"\nPrime 10 righe test set per rm_id={sample_rm}:")
print(test_sample[feature_cols + ['date', 'predicted_daily_weight']])

# Feature del training set per questo rm_id (ultime 10 righe)
train_sample = train_df[train_df['rm_id'] == sample_rm].tail(10)
print(f"\nUltime 10 righe training set per rm_id={sample_rm}:")
print(train_sample[feature_cols + ['date', 'net_weight']])

# Statistiche delle feature nel test vs training
print(f"\n=== Confronto statistiche feature test vs training ===")
for col in feature_cols:
    test_mean = final_test_df[col].mean()
    test_std = final_test_df[col].std()
    train_mean = train_df[col].mean()
    train_std = train_df[col].std()
    print(f"{col:20s} - Test: mean={test_mean:8.2f}, std={test_std:8.2f} | Train: mean={train_mean:8.2f}, std={train_std:8.2f}")

# Verifica predizioni sul validation set
print(f"\n=== Predizioni sul validation set (2024) ===")
print(f"Range predizioni validation: [{val_pred.min():.4f}, {val_pred.max():.4f}]")
print(f"Predizioni > 0 nel validation: {(val_pred > 0).sum()} su {len(val_pred)} ({100*(val_pred > 0).sum()/len(val_pred):.2f}%)")
print(f"Mean predizioni validation: {val_pred.mean():.4f}")
print(f"Median predizioni validation: {np.median(val_pred):.4f}")

# Confronta con le predizioni del test 2025
print(f"\n=== Predizioni sul test set (2025) ===")
print(f"Range predizioni test: [{daily_predictions.min():.4f}, {daily_predictions.max():.4f}]")
print(f"Predizioni > 0 nel test: {(daily_predictions > 0).sum()} su {len(daily_predictions)} ({100*(daily_predictions > 0).sum()/len(daily_predictions):.2f}%)")
print(f"Mean predizioni test: {daily_predictions.mean():.4f}")
print(f"Median predizioni test: {np.median(daily_predictions):.4f}")

Debug approfondito: analisi feature del test set...

=== Analisi rm_id=2561.0 ===

Prime 10 righe test set per rm_id=2561.0:
         year  month  day  dayofweek  dayofyear  weekofyear  quarter  lag_7d  \
1033409  2025      1    1          2          1           1        1     0.0   
1033410  2025      1    2          3          2           1        1     0.0   
1033411  2025      1    3          4          3           1        1     0.0   
1033412  2025      1    4          5          4           1        1     0.0   
1033413  2025      1    5          6          5           1        1     0.0   
1033414  2025      1    6          0          6           2        1     0.0   
1033415  2025      1    7          1          7           2        1     0.0   
1033416  2025      1    8          2          8           2        1     0.0   
1033417  2025      1    9          3          9           2        1     0.0   
1033418  2025      1   10          4         10           2        1     0.

In [27]:
print("=== ANALISI CRITICA: Cosa stiamo predicendo? ===\n")

# 1. Verifica cosa chiede prediction_mapping
print("1. Cosa chiede prediction_mapping.csv?")
print(f"   Colonne: {prediction_mapping_df.columns.tolist()}")
print(f"\n   Prime 5 righe:")
print(prediction_mapping_df.head())

# 2. Verifica date in prediction_mapping
print(f"\n2. Date richieste per le predizioni:")
pred_dates = pd.to_datetime(prediction_mapping_df['forecast_end_date'])
print(f"   Range: {pred_dates.min()} to {pred_dates.max()}")
print(f"   Unique dates: {pred_dates.nunique()}")

# 3. Il problema chiede CUMULATIVE weight fino a forecast_end_date
# Verifichiamo se nel training set abbiamo questa informazione
print(f"\n3. Nel training set abbiamo 'net_weight' (peso giornaliero)")
print(f"   Ma prediction_mapping chiede il peso CUMULATIVO dal 1 Gennaio")

# 4. Vediamo un esempio concreto
example_rm = 365
example_mapping = prediction_mapping_df[prediction_mapping_df['rm_id'] == example_rm].head(10)
print(f"\n4. Esempio per rm_id={example_rm} in prediction_mapping:")
print(example_mapping[['ID', 'rm_id', 'forecast_start_date', 'forecast_end_date']])

# 5. Nel training set, possiamo calcolare pesi cumulativi storici?
print(f"\n5. Calcolo pesi cumulativi storici per rm_id={example_rm} (ultimi 30 giorni 2024):")
example_train = master_df[
    (master_df['rm_id'] == example_rm) & 
    (master_df['date'] >= '2024-12-01')
][['date', 'net_weight']].tail(30)
example_train['cumulative_weight'] = example_train['net_weight'].cumsum()
print(example_train)

print("\n" + "="*80)
print("CONCLUSIONE:")
print("- Nel training set abbiamo 'net_weight' (peso giornaliero)")
print("- Stiamo predicendo 'predicted_daily_weight' (peso giornaliero futuro)")  
print("- Poi calcoliamo 'predicted_cumulative_weight' per la submission")
print("- Ma forse dovremmo usare 'cumulative_weight' come target nel training!")
print("="*80)

=== ANALISI CRITICA: Cosa stiamo predicendo? ===

1. Cosa chiede prediction_mapping.csv?
   Colonne: ['ID', 'rm_id', 'forecast_start_date', 'forecast_end_date']

   Prime 5 righe:
   ID  rm_id forecast_start_date forecast_end_date
0   1    365          2025-01-01        2025-01-02
1   2    365          2025-01-01        2025-01-03
2   3    365          2025-01-01        2025-01-04
3   4    365          2025-01-01        2025-01-05
4   5    365          2025-01-01        2025-01-06

2. Date richieste per le predizioni:
   Range: 2025-01-02 00:00:00 to 2025-05-31 00:00:00
   Unique dates: 150

3. Nel training set abbiamo 'net_weight' (peso giornaliero)
   Ma prediction_mapping chiede il peso CUMULATIVO dal 1 Gennaio

4. Esempio per rm_id=365 in prediction_mapping:
   ID  rm_id forecast_start_date forecast_end_date
0   1    365          2025-01-01        2025-01-02
1   2    365          2025-01-01        2025-01-03
2   3    365          2025-01-01        2025-01-04
3   4    365          2

In [26]:
print("Creazione file di submission...")

# Caricamento prediction mapping
submission_df = prediction_mapping_df.copy()
# Converti forecast_end_date in datetime e rinominala come 'date' per il merge
submission_df['date'] = pd.to_datetime(submission_df['forecast_end_date'])

# Merge con le predizioni cumulative
submission_df = pd.merge(
    submission_df,
    final_test_df[['rm_id', 'date', 'predicted_cumulative_weight']],
    on=['rm_id', 'date'],
    how='left'
)

# Gestione eventuali valori mancanti
submission_df['predicted_cumulative_weight'] = submission_df['predicted_cumulative_weight'].fillna(0)

# Formato finale (usa 'ID' con maiuscolo come nel file originale)
final_submission = submission_df[['ID', 'predicted_cumulative_weight']].copy()
final_submission.columns = ['ID', 'predicted_weight']

# Salvataggio
final_submission.to_csv('submission.csv', index=False)

print(f"\nFile 'submission.csv' creato con successo!")
print(f"Numero di predizioni: {len(final_submission)}")
print(f"\nPrime 10 righe del file di submission:")
print(final_submission.head(10))
print(f"\nStatistiche predizioni:")
print(final_submission['predicted_weight'].describe())

Creazione file di submission...

File 'submission.csv' creato con successo!
Numero di predizioni: 30450

Prime 10 righe del file di submission:
   ID  predicted_weight
0   1               0.0
1   2               0.0
2   3               0.0
3   4               0.0
4   5               0.0
5   6               0.0
6   7               0.0
7   8               0.0
8   9               0.0
9  10               0.0

Statistiche predizioni:
count    30450.000000
mean       336.035611
std       3408.655704
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max      48532.876214
Name: predicted_weight, dtype: float64
